In [ ]:
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)

symbol = "BTCUSDT"
market = "spot"


In [ ]:
# Fetch 5m data
df1 = fetch_binance_klines(symbol, "1m", start="2026-01-01", end=None, market=market)

In [ ]:
# Indicators (use clean tag; df columns become rsi14__RSI, rsi14__BULL_DIV, etc.)
ind_5m = [
    IndicatorSpec("rsi_divergence", tag="rsi14", config={
        "length": 14,
        "pivot_lookback": 5,
        "min_rsi_delta": 2,

        # divergence filtering
        "os_level": 30,
        "ob_level": 70,
        "zone_mode": "cross",

        # panel styling
        "major_levels": [30, 70],
        "minor_levels": [20, 80],
        "show_zone_shading": True,

        # label tuning
        "show_div_labels": True,
        "max_div_labels": 6,
        "label_font_size": 10,
        "label_xshift": 10,
        "label_yshift": 14,

        # main chart markers
        "mark_price": True,
        "price_marker_size": 20,
        "marker_offset_mult": 1.35,
    }),
]

df1_feat, _, _ = build_feature_df(df1, tz="Asia/Karachi", ma_windows=[20], indicators=ind_5m)

In [ ]:
# --- LONG OPEN: cross up 20 after being under 20
open_rules_long = ALL(
    Rule("Prev5 RSI < 20", lambda c: c.prev_all_below("rsi14__RSI", 20, n=5)),
    Rule("RSI cross up 20", lambda c: c.cross_up("rsi14__RSI", 20, lookback=1)),
)

# --- LONG CLOSE: hit (or cross into) overbought
close_rules_long = ALL(
    Rule("RSI >= 80", lambda c: c.v("rsi14__RSI") >= 80),
    # or: Rule("RSI cross up 80", lambda c: c.cross_up("rsi14__RSI", 80, lookback=1)),
)

# --- SHORT OPEN: opposite (overbought -> cross down 80)
open_rules_short = ALL(
    Rule("Prev5 RSI > 80", lambda c: c.prev_all_above("rsi14__RSI", 80, n=5)),
    Rule("RSI cross down 80", lambda c: c.cross_down("rsi14__RSI", 80, lookback=1)),
)

# --- SHORT CLOSE: opposite (oversold)
close_rules_short = ALL(
    Rule("RSI <= 20", lambda c: c.v("rsi14__RSI") <= 20),
    # or: Rule("RSI cross down 20", lambda c: c.cross_down("rsi14__RSI", 20, lookback=1)),
)

strategy = Strategy(
    open_rules_long=open_rules_long,
    close_rules_long=close_rules_long,
    allow_short=True,
    open_rules_short=open_rules_short,
    close_rules_short=close_rules_short,
)

res = run_simulation(
    df=df1_feat,
    strategy=strategy,
    cfg=SimConfig(initial_cash=10_000, max_open_trades=1, fee_bps=5, slippage_bps=1),
    time_col="t",
    price_col="close",
)

In [ ]:
fig1 = plot_balance_by_trade(res.trades, initial_cash=10_000, interval="5m")
fig1.show()

fig2 = plot_trade_pnl_bars(res.trades, initial_cash=10_000, interval="5m")
fig2.show()

trade_summary_table(res.trades, initial_cash=10_000, interval="5m")

res.stats#, res.events.head(), res.equity_curve.tail()

In [ ]:
plot_simulation(
    df_raw=df1_feat,  # (ideally raw candles df; but if yours works as-is, ok)
    symbol=symbol,
    interval="5m",
    market=market,
    trades=res.trades,
    ma_windows=[20],
    indicators=ind_5m,
    plot_cfg=PlotConfig(
        tz="Asia/Karachi",
        date_from="2026-02-01",
        date_to="2026-02-05",
        height=1400,
        width=1900
    ),
)

In [ ]:

# 1m features
df1 = fetch_binance_klines(symbol, "15m", start="2026-03-1", end=None, market=market)
ind_1m = [
    IndicatorSpec("rsi_divergence", tag="rsi14", config={"length": 14}),
    IndicatorSpec("stochastic", tag="st14", config={"k_length": 14, "d_length": 3, "smooth": 3}),
    IndicatorSpec("supertrend", tag="st10_3", config={"length": 10, "multiplier": 3}),
]
df1_feat, _, _ = build_feature_df(df1, tz="Asia/Karachi", ma_windows=[20], indicators=ind_1m)

# 5m features
df5 = fetch_binance_klines(symbol, "5m", start="2026-02-15", end=None, market=market)
ind_5m = [
    IndicatorSpec("rsi_divergence", tag="rsi14", config={"length": 14}),
    IndicatorSpec("stochastic", tag="st14", config={"k_length": 14, "d_length": 3, "smooth": 3}),
]
df5_feat, _, _ = build_feature_df(df5, tz="Asia/Karachi", ma_windows=[], indicators=ind_5m)

# Merge MTF into base (1m)
merged = align_timeframes(df1_feat, {"5m": df5_feat}, base_label="1m")

# Define rules:
open_1m = ALL(
    Rule("1m_rsi>55", lambda r: r["1m__rsi14__RSI"] > 55),
    Rule("1m_stoch_K>D", lambda r: r["1m__st14__K"] > r["1m__st14__D"]),
    Rule("1m_ST_buy_flip", lambda r: bool(r["1m__st10_3__BUY"]) if "1m__st10_3__BUY" in r else False),
)

open_5m = ALL(
    Rule("5m_rsi>55", lambda r: r["5m__rsi14__RSI"] > 55),
    Rule("5m_stoch_K>D", lambda r: r["5m__st14__K"] > r["5m__st14__D"]),
)

open_rules = ANY(open_1m, open_5m)  # OR between timeframes

close_rules = ANY(
    Rule("1m_rsi<45", lambda r: r["1m__rsi14__RSI"] < 45),
    Rule("1m_ST_sell_flip", lambda r: bool(r["1m__st10_3__SELL"]) if "1m__st10_3__SELL" in r else False),
)

strategy = Strategy(open_rules_long=open_rules, close_rules_long=close_rules)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=SimConfig(initial_cash=10_000, max_open_trades=2, fee_bps=5, slippage_bps=1),
    time_col="t",
    price_col="close",
)

res.stats, res.events.head(), res.equity_curve.tail()

In [ ]:
plot_simulation(
    df_raw=df1,  # raw 1m df for candles
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[20],
    indicators=ind_1m,
    plot_cfg=PlotConfig(tz="Asia/Karachi", show_last="2d", height=1400, width=1900),
)